In [1]:
import sys
sys.path.append('../')

from gears import PertData, GEARS

Load data. We use norman as an example.

In [2]:
pert_data = PertData('./data')
pert_data.load(data_name = 'norman')
pert_data.prepare_split(split = 'simulation', seed = 1)
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
7
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!


here1


In [3]:
len(pert_data.pert_names.tolist())

9853

Create a model object; if you use [wandb](https://wandb.ai), you can easily track model training and evaluation by setting `weight_bias_track` to true, and specify the `proj_name` and `exp_name` that you like.

In [3]:
gears_model = GEARS(pert_data, device = 'cuda:0', 
                        weight_bias_track = False, 
                        proj_name = 'gt_gears_contra_pretaining',
                        exp_name = '2layers')
gears_model.model_initialize(hidden_size = 64)


Found local copy...


In [5]:
# --- Add this line ---
gears_model.pretrain_embeddings(pretrain_epochs=15, pretrain_lr=5e-4, temperature=0.1, num_negatives=64, positive_threshold=0.3) # Adjust params as needed
# --------------------

Starting contrastive pre-training of gene embeddings...
Calculating mean delta expression for single perturbations...


(Count: 148)


Calculating Deltas: 100%|██████████| 148/148 [00:01<00:00, 78.73it/s]
Finished calculating deltas.
Calculating similarity matrix...
Finished calculating similarity matrix.
Processed 148 items from valid_pert_names:
  - Started with 'ctrl+': 47
  - Ended with '+ctrl': 101
Extracted 102 unique pure gene names.
Pre-training embeddings for 102 genes based on perturbation similarity.
['ctrl+CEBPE', 'GLB1L2+ctrl', 'ctrl+ETS2', 'LHX1+ctrl', 'COL2A1+ctrl', 'MIDN+ctrl', 'DLX2+ctrl', 'HES7+ctrl', 'FOXO4+ctrl', 'ctrl+CBFA2T3', 'ctrl+RUNX1T1', 'CELF2+ctrl', 'RUNX1T1+ctrl', 'FOSB+ctrl', 'MAPK1+ctrl', 'HOXB9+ctrl', 'ctrl+HOXB9', 'ETS2+ctrl', 'FOXA3+ctrl', 'CEBPE+ctrl', 'COL1A1+ctrl', 'FOXF1+ctrl', 'C3orf72+ctrl', 'ctrl+DLX2', 'FOXL2+ctrl', 'ctrl+CEBPA', 'HNF4A+ctrl', 'ISL2+ctrl', 'ctrl+COL2A1', 'OSR2+ctrl', 'ctrl+SPI1', 'CEBPB+ctrl', 'CEBPA+ctrl', 'NIT1+ctrl', 'ZBTB10+ctrl', 'ctrl+MAPK1', 'ctrl+ISL2', 'ctrl+SNAI1', 'ctrl+CEBPB', 'HOXA13+ctrl', 'CITED1+ctrl', 'ctrl+FOXF1', 'ctrl+OSR2', 'ctrl+FOXL2', 

You can find available tunable parameters in model_initialize via

In [6]:
gears_model.tunable_parameters()

{'hidden_size': 'hidden dimension, default 64',
 'num_go_gnn_layers': 'number of GNN layers for GO graph, default 1',
 'num_gene_gnn_layers': 'number of GNN layers for co-expression gene graph, default 1',
 'decoder_hidden_size': 'hidden dimension for gene-specific decoder, default 16',
 'num_similar_genes_go_graph': 'number of maximum similar K genes in the GO graph, default 20',
 'num_similar_genes_co_express_graph': 'number of maximum similar K genes in the co expression graph, default 20',
 'coexpress_threshold': 'pearson correlation threshold when constructing coexpression graph, default 0.4',
 'uncertainty': 'whether or not to turn on uncertainty mode, default False',
 'uncertainty_reg': 'regularization term to balance uncertainty loss and prediction loss, default 1',
 'direction_lambda': 'regularization term to balance direction loss and prediction loss, default 1'}

Train your model:

Note: For the sake of demo, we set epoch size to 1. To get full model, set `epochs = 20`.

In [7]:
gears_model.train(epochs = 20,lr= 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.4686
Epoch 1 Step 51 Train Loss: 0.4949
Epoch 1 Step 101 Train Loss: 0.4441
Epoch 1 Step 151 Train Loss: 0.5526
Epoch 1 Step 201 Train Loss: 0.5809
Epoch 1 Step 251 Train Loss: 0.4552
Epoch 1 Step 301 Train Loss: 0.5374
Epoch 1 Step 351 Train Loss: 0.4854
Epoch 1 Step 401 Train Loss: 0.4079
Epoch 1 Step 451 Train Loss: 0.4950
Epoch 1 Step 501 Train Loss: 0.4909
Epoch 1 Step 551 Train Loss: 0.5055
Epoch 1 Step 601 Train Loss: 0.4850
Epoch 1 Step 651 Train Loss: 0.5023
Epoch 1 Step 701 Train Loss: 0.4390
Epoch 1 Step 751 Train Loss: 0.5342
Epoch 1 Step 801 Train Loss: 0.4462
Epoch 1 Step 851 Train Loss: 0.5612
Epoch 1 Step 901 Train Loss: 0.5627
Epoch 1 Step 951 Train Loss: 0.4087
Epoch 1 Step 1001 Train Loss: 0.4138
Epoch 1 Step 1051 Train Loss: 0.4744
Epoch 1 Step 1101 Train Loss: 0.4260
Epoch 1 Step 1151 Train Loss: 0.4910
Epoch 1 Step 1201 Train Loss: 0.4827
Epoch 1 Step 1251 Train Loss: 0.4361
Epoch 1 Step 1301 Train Loss: 0.5041
Epoch 

Save and load pretrained models:

In [8]:
gears_model.save_model('gt_contra_pretraining')
gears_model.load_pretrained('gt_contra_pretraining')

Make prediction for new perturbation:

In [10]:
gears_model = GEARS(pert_data, device = 'cuda:0', 
                        weight_bias_track = False,
                        proj_name = 'gt_gears_contra_pretaining',
                        exp_name = '2layers')
gears_model.model_initialize(hidden_size = 64, use_transformer = False)

Found local copy...


In [11]:
gears_model.train(epochs = 20,lr= 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.4454
Epoch 1 Step 51 Train Loss: 0.5241
Epoch 1 Step 101 Train Loss: 0.4362
Epoch 1 Step 151 Train Loss: 0.5678
Epoch 1 Step 201 Train Loss: 0.5279
Epoch 1 Step 251 Train Loss: 0.5607
Epoch 1 Step 301 Train Loss: 0.4564
Epoch 1 Step 351 Train Loss: 0.5219
Epoch 1 Step 401 Train Loss: 0.4661
Epoch 1 Step 451 Train Loss: 0.5658
Epoch 1 Step 501 Train Loss: 0.4328
Epoch 1 Step 551 Train Loss: 0.5215
Epoch 1 Step 601 Train Loss: 0.4239
Epoch 1 Step 651 Train Loss: 0.4975
Epoch 1 Step 701 Train Loss: 0.4729
Epoch 1 Step 751 Train Loss: 0.4551
Epoch 1 Step 801 Train Loss: 0.4920
Epoch 1 Step 851 Train Loss: 0.4641
Epoch 1 Step 901 Train Loss: 0.4891
Epoch 1 Step 951 Train Loss: 0.4958
Epoch 1 Step 1001 Train Loss: 0.4215
Epoch 1 Step 1051 Train Loss: 0.4199
Epoch 1 Step 1101 Train Loss: 0.4585
Epoch 1 Step 1151 Train Loss: 0.5607
Epoch 1 Step 1201 Train Loss: 0.5067
Epoch 1 Step 1251 Train Loss: 0.5259
Epoch 1 Step 1301 Train Loss: 0.4736
Epoch 

In [7]:
gears_model.predict([['FEV'], ['FEV', 'AHR']])

{'FEV': array([-1.5115363e-06,  4.4304952e-02,  1.0309354e-01, ...,
         3.3967001e+00,  7.8529231e-03,  1.0920237e-31], dtype=float32),
 'FEV_SAMD11': array([-2.2916190e-06,  9.7577907e-02,  1.6493453e-01, ...,
         3.2082996e+00,  7.6769367e-03,  1.7619579e-31], dtype=float32)}

In [5]:
# --- Add this line ---
gears_model.pretrain_embeddings(pretrain_epochs=50, pretrain_lr=5e-4, temperature=0.1, num_negatives=64, positive_threshold=0.4) # Adjust params as needed
# --------------------

Starting contrastive pre-training of gene embeddings...
Calculating mean delta expression for single perturbations...


(Count: 148)


Calculating Deltas: 100%|██████████| 148/148 [00:01<00:00, 85.11it/s]
Finished calculating deltas.
Calculating similarity matrix...
Finished calculating similarity matrix.
Processed 148 items from valid_pert_names:
  - Started with 'ctrl+': 47
  - Ended with '+ctrl': 101
Extracted 102 unique pure gene names.
Pre-training embeddings for 102 genes based on perturbation similarity.
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
Pre-train Epoch 1/50, Avg Loss: 0.3224
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable positive found
# Skip if no suitable po

In [8]:
gears_model.train(epochs = 20,lr= 1e-3)
gears_model.save_model('gt_contra_pretraining_0.4')


Start Training...


Epoch 1 Step 1 Train Loss: 0.4395
Epoch 1 Step 51 Train Loss: 0.5053
Epoch 1 Step 101 Train Loss: 0.4818
Epoch 1 Step 151 Train Loss: 0.5080
Epoch 1 Step 201 Train Loss: 0.4539
Epoch 1 Step 251 Train Loss: 0.5737
Epoch 1 Step 301 Train Loss: 0.5265
Epoch 1 Step 351 Train Loss: 0.4885
Epoch 1 Step 401 Train Loss: 0.5052
Epoch 1 Step 451 Train Loss: 0.5116
Epoch 1 Step 501 Train Loss: 0.4481
Epoch 1 Step 551 Train Loss: 0.5216
Epoch 1 Step 601 Train Loss: 0.4407
Epoch 1 Step 651 Train Loss: 0.6527
Epoch 1 Step 701 Train Loss: 0.5350
Epoch 1 Step 751 Train Loss: 0.4478
Epoch 1 Step 801 Train Loss: 0.5189
Epoch 1 Step 851 Train Loss: 0.4911
Epoch 1 Step 901 Train Loss: 0.5675
Epoch 1 Step 951 Train Loss: 0.4320
Epoch 1 Step 1001 Train Loss: 0.4714
Epoch 1 Step 1051 Train Loss: 0.4277
Epoch 1 Step 1101 Train Loss: 0.5458
Epoch 1 Step 1151 Train Loss: 0.4481
Epoch 1 Step 1201 Train Loss: 0.4785
Epoch 1 Step 1251 Train Loss: 0.4848
Epoch 1 Step 1301 Train Loss: 0.4930
Epoch 1 Step 1351 Train 

In [9]:
gears_model = GEARS(pert_data, device = 'cuda:0', 
                        weight_bias_track = False,
                        proj_name = 'gt_gears_contra_pretaining',
                        exp_name = '2layers')
gears_model.model_initialize(hidden_size = 64, use_transformer = False)

gears_model.train(epochs = 20,lr= 1e-3)
gears_model.save_model('contra_pretraining_0.4')


Found local copy...
Start Training...
Epoch 1 Step 1 Train Loss: 0.4663
Epoch 1 Step 51 Train Loss: 0.5779
Epoch 1 Step 101 Train Loss: 0.4666
Epoch 1 Step 151 Train Loss: 0.4099
Epoch 1 Step 201 Train Loss: 0.4820
Epoch 1 Step 251 Train Loss: 0.4253
Epoch 1 Step 301 Train Loss: 0.4434
Epoch 1 Step 351 Train Loss: 0.4640
Epoch 1 Step 401 Train Loss: 0.5633
Epoch 1 Step 451 Train Loss: 0.4304
Epoch 1 Step 501 Train Loss: 0.4594
Epoch 1 Step 551 Train Loss: 0.4705
Epoch 1 Step 601 Train Loss: 0.5301
Epoch 1 Step 651 Train Loss: 0.4623
Epoch 1 Step 701 Train Loss: 0.5329
Epoch 1 Step 751 Train Loss: 0.5155
Epoch 1 Step 801 Train Loss: 0.4422
Epoch 1 Step 851 Train Loss: 0.4027
Epoch 1 Step 901 Train Loss: 0.4271
Epoch 1 Step 951 Train Loss: 0.4498
Epoch 1 Step 1001 Train Loss: 0.4526
Epoch 1 Step 1051 Train Loss: 0.4344
Epoch 1 Step 1101 Train Loss: 0.4483
Epoch 1 Step 1151 Train Loss: 0.5195
Epoch 1 Step 1201 Train Loss: 0.4812
Epoch 1 Step 1251 Train Loss: 0.4715
Epoch 1 Step 1301 Train

Gene list can be found here:

In [8]:
gears_model.gene_list[:5]

['RP11-34P13.8', 'RP11-54O7.3', 'SAMD11', 'PERM1', 'HES4']